In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.dummy import DummyRegressor
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.impute import SimpleImputer
from scipy import stats
from tqdm import tqdm
import time
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
import os
from datetime import datetime
import openpyxl
 

##machine learning model prediction

In [2]:

# 是否进行验证
isVerify = True
# 是否保存Excel
isSaveExcel = True
# 是否进行置信区间分析
isConfidence = True
# 是否进行预测
isPrediction = True  


set text format

In [3]:
# 设置中文字体
mpl.rcParams['font.sans-serif'] = ['SimHei']
mpl.rcParams['axes.unicode_minus'] = False

parse_flavors method and transform to dict file

In [4]:
def parse_flavors(flavor_string):
    """
    解析 'flavors' 列，将其转换为字典。
    """
    flavor_dict = {}
    if isinstance(flavor_string, str):
        pairs = flavor_string.split(', ')
        for pair in pairs:
            if ':' in pair:
                key, value = pair.split(':', 1)
                flavor_dict[key.strip()] = value.strip()
            else:
                flavor_dict[pair.strip()] = 'True'
    return flavor_dict


load_and_preprocess_data

In [6]:
def load_and_preprocess_data(file_path):
    """
    加载并预处理销售数据。

    这个函数主要完成以下任务：
    1. 从指定的CSV文件中加载数据
    2. 解析并展开 'flavors' 列
    3. 处理日期时间信息，提取年、月、日、星期和小时

    参数:
    file_path (str): CSV文件的路径

    返回:
    pandas.DataFrame: 预处理后的数据框

    注意:
    - 此函数会修改输入的DataFrame
    - 'flavors' 列会被解析并展开为多个新列
    - 日期时间信息会被拆分为多个新列
    """
    print("开始加载和预处理数据...")
    df = pd.read_csv(file_path)
    print(f"成功加载数据，共 {len(df)} 行")
    df['flavors'] = df['flavors'].fillna('')
    flavor_df = df['flavors'].apply(parse_flavors).apply(pd.Series)
    df = pd.concat([df, flavor_df], axis=1)
    df = df.drop('flavors', axis=1)
    
    # 处理
    df['date_paid'] = pd.to_datetime(df['date_paid'], format='%Y/%m/%d %H:%M', errors='coerce')
    df['year'] = df['date_paid'].dt.year
    df['month'] = df['date_paid'].dt.month
    df['day'] = df['date_paid'].dt.day
    df['weekday'] = df['date_paid'].dt.weekday
    df['hour'] = df['date_paid'].dt.hour
    
    print("数据预处理完成")
    return df

engineer_features

In [7]:
def engineer_features(df):
    """
    进行特征工程，处理和转换数据集中的特征。

    这个函数主要完成以下任务：
    1. 定义要使用的特征列表
    2. 对分类特征进行标签编码
    3. 处理数值型特征的缺失值，并进行标准化
    4. 处理分类特征的缺失值
    5. 处理目标变量 'total' 的缺失值

    参数:
    df (pandas.DataFrame): 包含原始数据的DataFrame

    返回:
    tuple: 包含以下元素的元组
        - 处理后的特征 (pandas.DataFrame)
        - 目标变量 'total' (pandas.Series)
        - 原始的咖啡名称 (pandas.Series)

    注意:
    - 此函数会修改输入的DataFrame
    - 返回的特征DataFrame只包含选定的特征列
    """
    print("开始特征工程...")
    features = ['name', 'base_price', 'total_price_calculated', 'size', 'Milk', 'Strength', 'Decaf', 'Temp', 'Honey', 'Syrup', 'White Sugar', 'Raw Sugar', 'Equal Sugar', 'Extra shot', 'year', 'month', 'day', 'weekday', 'hour']
    
    le = LabelEncoder()
    df_encoded = df.copy()
    for col in features:
        if col in df.columns and df[col].dtype == 'object':
            df_encoded[col] = le.fit_transform(df[col].astype(str))
    
    # 处理数值型特征的缺失值
    numeric_features = df_encoded[features].select_dtypes(include=['int64', 'float64']).columns
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    df_encoded[numeric_features] = scaler.fit_transform(imputer.fit_transform(df_encoded[numeric_features]))
    
    # 处理分类特征的缺失值
    categorical_features = df_encoded[features].select_dtypes(include=['object']).columns
    df_encoded[categorical_features] = df_encoded[categorical_features].fillna('Unknown')
    
    # 处理目标变量 'total' 的缺失值
    df_encoded['total'] = df_encoded['total'].fillna(df_encoded['total'].mean())
    
    # 保存原始特征值到编码值的映射
    feature_mappings = {}
    for col in features:
        if df[col].dtype == 'object':
            feature_mappings[col] = dict(zip(le.fit_transform(df[col].astype(str)), df[col].astype(str)))
    
    print("特征工程完成")
    return df_encoded[features], df_encoded['total'], df['name'], feature_mappings

trian and evlaute model

In [8]:
def train_and_evaluate_model(X_train, X_test, y_train, y_test):
    """
    训练和评估模型。
    """
    print("开始训练模型...")
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    
    print("型训练完成，开始评估...")
    y_pred = rf_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"均方误差: {mse}")
    print(f"R平方分数: {r2}")
    print(f"平均绝对误差: {mae}")
    
    if isVerify:
        # 交叉验证
        print("开始交叉验证...")
        cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5)
        print(f"交叉验证分数: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        # 与基准模型比较
        dummy_regr = DummyRegressor(strategy="mean")
        dummy_regr.fit(X_train, y_train)
        dummy_pred = dummy_regr.predict(X_test)
        dummy_mse = mean_squared_error(y_test, dummy_pred)
        print(f"基准模型均方误差: {dummy_mse}")
        
        # 残差分析
        residuals = y_test - y_pred
        plt.figure(figsize=(10, 6))
        plt.scatter(y_pred, residuals)
        plt.title('残差图')
        plt.xlabel('预测值')
        plt.ylabel('残差')
        plt.savefig('残差图.png')
        plt.close()
        
        # 残差的正态性检验
        _, p_value = stats.normaltest(residuals)
        print(f"残差正态性检验 p-value: {p_value}")
    
    return rf_model

plot_feature_importance method

In [11]:
def plot_feature_importance(model, features, X, y):
    """
    绘制特征重要性图。

    参数:
    model (sklearn.ensemble.RandomForestRegressor): 训练好的随机森林回归模型
    features (list): 特征名称列表
    X (pandas.DataFrame): 特征数据
    y (pandas.Series): 目标变量
    """ 
    print("开始绘制特征重要性图...")
    feature_importance = model.feature_importances_
    feature_importance_df = pd.DataFrame({'feature': features, 'importance': feature_importance})
    feature_importance_df = feature_importance_df.sort_values('importance', ascending=False)

    plt.figure(figsize=(14, 10))  # 增加图像大小以适应更多特征
    plt.bar(feature_importance_df['feature'], feature_importance_df['importance'])
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    plt.title('总销售额与各特征的正相关性', fontsize=16)
    plt.xlabel('特征', fontsize=14)
    plt.ylabel('正相关性', fontsize=14)
    plt.tight_layout()
    plt.savefig('特征正相关性图.png', dpi=300)
    print("特征正相关性图绘制完成")

    # 添加 bootstrap 置信区间
    if isConfidence:
        print("开始计算 bootstrap 置信区间...")
        n_bootstraps = 100  # 减少bootstrap次数
        n_features = len(features)
        bootstrapped_importances = np.zeros((n_bootstraps, n_features))
        
        # 使用joblib进行并行计算
        from joblib import Parallel, delayed
        
        def bootstrap_iteration(i, X, y):
            np.random.seed(i)
            bootstrap_indices = np.random.choice(len(X), len(X), replace=True)
            bootstrap_X = X.iloc[bootstrap_indices]
            bootstrap_y = y.iloc[bootstrap_indices]
            bootstrap_model = RandomForestRegressor(n_estimators=50, random_state=i)  # 减少树的数量
            bootstrap_model.fit(bootstrap_X, bootstrap_y)
            return bootstrap_model.feature_importances_
        
        bootstrapped_importances = Parallel(n_jobs=-1)(
            delayed(bootstrap_iteration)(i, X, y) for i in range(n_bootstraps)
        )
        
        print("计算置信区间...")
        confidence_intervals = np.percentile(bootstrapped_importances, [2.5, 97.5], axis=0)
        print("Bootstrap 置信区间计算完成")
        plt.figure(figsize=(14, 10))
        plt.errorbar(range(n_features), feature_importance, 
                    yerr=[feature_importance - confidence_intervals[0], 
                        confidence_intervals[1] - feature_importance],
                    fmt='o', capsize=5)
        plt.xticks(range(n_features), features, rotation=90)
        plt.title('特征重要性及其95%置信区间')
        plt.xlabel('特征')
        plt.ylabel('重要性')
        plt.tight_layout()
        plt.savefig('特征重要性及其95%置信区间.png', dpi=300)
        plt.close()
        print("特征重要性及其95%置信区间图已保存")

analyze_product_performance method

In [12]:
def analyze_product_performance(df):
    """
    分析产品性能,找出表现最好和最差的产品。
    """
    print("开始分析产品性能...")
    product_performance = df.groupby('name').agg({
        'total': 'sum',
        'base_price': 'mean',
        'total_price_calculated': 'mean',
        'size': lambda x: x.mode().iloc[0] if not x.mode().empty else None,
        'Milk': lambda x: x.mode().iloc[0] if not x.mode().empty else None
    }).sort_values('total', ascending=False)
    
    # 重置索引，将产品名称作为一个列
    product_performance = product_performance.reset_index()
    
    top_10 = product_performance.head(10)
    bottom_10 = product_performance.tail(10)
    
    print("\n表现最好的10种产品:")
    for _, row in top_10.iterrows():
        print(f"产品名称: {row['name']}", end=',')
        print(f"  总销售额: {row['total']:.2f}", end=',')
        print(f"  平均基础价格: {row['base_price']:.2f}", end=',')
        print(f"  平均实际价格: {row['total_price_calculated']:.2f}", end=',')
        print(f"  常见尺寸: {row['size']}", end=',')
        print(f"  常见牛奶类型: {row['Milk']}")
    
    print("\n表现最差的10种产品:")
    for _, row in bottom_10.iterrows():
        print(f"产品名称: {row['name']}", end=',')
        print(f"  总销售额: {row['total']:.2f}", end=',')
        print(f"  平均基础价格: {row['base_price']:.2f}", end=',')
        print(f"  平均实际价格: {row['total_price_calculated']:.2f}", end=',')
        print(f"  常见尺寸: {row['size']}", end=',')
        print(f"  常见牛奶类型: {row['Milk']}")
    
    print("产品性能分析完成")
    return product_performance, top_10, bottom_10

analyze_customer_behavior method

In [13]:
def analyze_customer_behavior(df):
    """
    分析客户行为,找出新客户、老客户和只买过一次的产品。
    """
    print("开始分析客户行为...")
    
    customer_stats = df.groupby('customer_id').agg({
        'order_id': 'nunique',
        'name': 'count'
    }).rename(columns={'order_id': 'purchase_count', 'name': 'item_count'})
    
    new_customers = customer_stats[customer_stats['purchase_count'] == 1].index
    old_customers = customer_stats[customer_stats['item_count'] > 5].index
    
    print(f"新客户数量: {len(new_customers)}")
    print(f"老客户数量: {len(old_customers)}")
    
    one_time_purchases = df.groupby('name').filter(lambda x: x['customer_id'].nunique() == 1)
    bad_products = one_time_purchases['name'].unique()
    
    print("\n只买过次的产品:")
    for product in bad_products:
        print(f"- {product}")
    
    print("客户行为分析完成")
    return new_customers, old_customers, bad_products

save_to_excel method

In [14]:
def save_to_excel(data_dict, filename='咖啡店销售分析报告.xlsx'):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name, ext = os.path.splitext(filename)
    new_filename = f"{base_name}_{timestamp}{ext}"
    
    wb = Workbook()
    wb.remove(wb.active)
    
    for sheet_name, df in data_dict.items():
        ws = wb.create_sheet(title=sheet_name)
        
        ws['A1'] = sheet_name
        ws['A1'].font = Font(bold=True, size=14)
        ws['A1'].alignment = Alignment(horizontal='center')
        ws.merge_cells('A1:E1')
        
        # 将DataFrame转换为列表，以避免可能的格式问题
        data = [df.columns.tolist()] + df.values.tolist()
        for row in data:
            ws.append(row)
        
        for column_cells in ws.columns:
            length = max(len(str(cell.value)) for cell in column_cells)
            ws.column_dimensions[get_column_letter(column_cells[0].column)].width = length + 2

    try:
        wb.save(new_filename)
        print(f"Excel文件已保存: {new_filename}")
    except PermissionError:
        print(f"无法保存文件 {new_filename}。请确保文件未被其他程序打开，并且您有写入权限。")
        documents_path = os.path.expanduser("~/Documents")
        alternative_filename = os.path.join(documents_path, new_filename)
        try:
            wb.save(alternative_filename)
            print(f"Excel文件已保存到替代位置: {alternative_filename}")
        except Exception as e:
            print(f"保存文件失败。错误: {str(e)}")
    except Exception as e:
        print(f"保存文件时发生错误: {str(e)}")

    # 验证保存的数据
    try:
        saved_wb = openpyxl.load_workbook(new_filename)
        for sheet_name in data_dict.keys():
            if sheet_name in saved_wb.sheetnames:
                saved_sheet = saved_wb[sheet_name]
                saved_rows = list(saved_sheet.values)
                # print(f"Sheet '{sheet_name}' 中保存的行数: {len(saved_rows) - 1}")  # 减去标题行
            else:
                print(f"警告: Sheet '{sheet_name}' 未在保存的文件中找到")
        saved_wb.close()
    except Exception as e:
        print(f"验证保存的数据发生错误: {str(e)}")

predict_custom_product method

In [15]:
def predict_custom_product(model, X, feature_names, feature_mappings):
    print("\n开始自定义产品预测...")
    custom_input = {}
    custom_input_display = {}  # 用于显示的字典
    
    for feature in feature_names:
        if feature in ['base_price', 'total_price_calculated', 'year', 'month', 'day', 'weekday', 'hour']:
            while True:
                try:
                    value = float(input(f"请输入 {feature} 的值: "))
                    custom_input[feature] = value
                    custom_input_display[feature] = value
                    break
                except ValueError:
                    print("请输入有效的数字。")
        elif feature in feature_mappings:
            original_values = list(feature_mappings[feature].values())
            print(f"\n{feature} 的可选值:")
            for i, value in enumerate(original_values):
                print(f"{i + 1}. {value}")
            while True:
                try:
                    choice = int(input(f"请选择 {feature} 的值 (输入对应的数字): "))
                    if 1 <= choice <= len(original_values):
                        custom_input[feature] = list(feature_mappings[feature].keys())[choice - 1]
                        custom_input_display[feature] = original_values[choice - 1]
                        break
                    else:
                        print("请输入有效的选项数字。")
                except ValueError:
                    print("请输入有效的数字。")
        else:
            unique_values = X[feature].unique()
            print(f"\n{feature} 的可选值:")
            for i, value in enumerate(unique_values):
                print(f"{i + 1}. {value}")
            while True:
                try:
                    choice = int(input(f"请选择 {feature} 的值 (输入对应的数字): "))
                    if 1 <= choice <= len(unique_values):
                        custom_input[feature] = unique_values[choice - 1]
                        custom_input_display[feature] = unique_values[choice - 1]
                        break
                    else:
                        print("请输入有效的选项数字。")
                except ValueError:
                    print("请输入有效的数字。")
    
    # 将自定义输入转换为模型可以使用的格式
    custom_df = pd.DataFrame([custom_input])
    custom_X = pd.get_dummies(custom_df, columns=[col for col in custom_df.columns if col not in ['base_price', 'total_price_calculated', 'year', 'month', 'day', 'weekday', 'hour']])
    
    # 确保自定义输入具有与训练数据相同的列
    for col in X.columns:
        if col not in custom_X.columns:
            custom_X[col] = 0
    custom_X = custom_X[X.columns]
    
    # 进行预测
    prediction = model.predict(custom_X)[0]  # 获取单个预测值
    
    print("\n预测结果:")
    for feature, value in custom_input_display.items():
        print(f"{feature}: {value}")
    print(f"预测销售额: {prediction:.2f}")
    
    # 计算预测区间
    predictions = []
    for _ in range(1000):
        predictions.append(model.predict(custom_X)[0])  # 获取单个预测值
    prediction_interval = np.percentile(predictions, [2.5, 97.5])
    print(f"95% 预测区间: [{prediction_interval[0]:.2f}, {prediction_interval[1]:.2f}]")


#Main method

In [16]:
def main():
    print("程序开始执行...")
    df = load_and_preprocess_data(r'C:\Users\YU244\咖啡数据处理\datasets/processed_product_with_total.csv')
    
    print("检查缺失值:")
    print(df.isnull().sum())
    
    print("检查 'total' 列的统计信息:")
    print(df['total'].describe())
    
    X, y, original_names, feature_mappings = engineer_features(df)
    
    print("特征工程后检查 y 的统计信息:")
    print(y.describe())
    
    print("开始划分训练集和测试集...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("数据集分完成")
    
    model = train_and_evaluate_model(X_train, X_test, y_train, y_test)
    
    # 修改这行，传入 X 和 y
    plot_feature_importance(model, X.columns, X, y)
    
    df_with_original_names = df.copy()
    df_with_original_names['name'] = original_names
    product_performance, top_10, bottom_10 = analyze_product_performance(df_with_original_names)
    
    new_customers, old_customers, bad_products = analyze_customer_behavior(df)
    
    if isPrediction:
        print("生成示例预测...")
        sample_input = X.iloc[0].to_frame().T
        sample_prediction = model.predict(sample_input)[0]  # 获取单个预测值

        # 计算预测区间
        predictions = []
        for _ in range(1000):
            predictions.append(model.predict(sample_input)[0])  # 获取单个预测值
        prediction_interval = np.percentile(predictions, [2.5, 97.5])
        
        # 样本预测
        print("\n样本预测:")
        for feature, value in sample_input.iloc[0].items():
            if feature in feature_mappings:
                print(f"{feature}: {feature_mappings[feature].get(value, value)}")
            else:
                print(f"{feature}: {value}")
        print(f"预测总销售额: {sample_prediction:.2f}")
        print(f"95% 预测区间: [{prediction_interval[0]:.2f}, {prediction_interval[1]:.2f}]")

        # 添加自定义产品预测
        predict_custom_product(model, X, X.columns, feature_mappings)

    print("程序执行完毕。")

    if isSaveExcel:
        customer_behavior_df = pd.DataFrame({
            '新客户数量': [len(new_customers)],
            '老客户数量': [len(old_customers)],
            '只买过一次的产品数量': [len(bad_products)]
        })
        
        # 创建只买过一次的产品列表
        bad_products_df = pd.DataFrame(bad_products, columns=['只买过一次的产品'])
        
        # 计算需要的空行数
        max_rows = max(len(customer_behavior_df), len(bad_products_df))
        
        # 添加空行到较短的DataFrame
        customer_behavior_df = customer_behavior_df.reindex(range(max_rows))
        bad_products_df = bad_products_df.reindex(range(max_rows))
        
        # 合并两个DataFrame
        combined_customer_analysis = pd.concat([customer_behavior_df, bad_products_df], axis=1)

        data_to_save = {
            '产品性能分析': product_performance,
            '表现最好的10种产品': top_10,
            '表现最差的10种产品': bottom_10,
            '客户行为分析': combined_customer_analysis
        }
        # print("data_to_save: \n", data_to_save)
        # print()
        save_to_excel(data_to_save)
    
    # 绘制残差图
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred
    plt.figure(figsize=(10, 6)) 
    plt.scatter(y_pred, residuals)
    plt.title('残差图')
    plt.xlabel('预测值')
    plt.ylabel('残差')
    plt.savefig('残差分析.png')
    plt.close()
    print("残差图已保存为: 残差分析.png")



#use all methods

In [ ]:

if __name__ == "__main__":
    main()



程序开始执行...
开始加载和预处理数据...
成功加载数据，共 70587 行
数据预处理完成
检查缺失值:
name                          1
base_price                    1
total_price_calculated        1
total                         1
customer_id                   1
date_paid                  2481
order_id                      1
Unnamed: 8                70587
Unnamed: 9                70587
Unnamed: 10               70586
size                          1
Milk                      17738
Strength                  20505
Decaf                     26808
Temp                      23373
Honey                     24972
Syrup                     20568
White Sugar               21343
Raw Sugar                 21343
Equal Sugar               21345
Extra shot                25309
Bread                     67701
Flavour                   67464
Extras                    64720
Cream                     69864
Heated                    70054
Toppings                  70558
Ingredients               69875
Options                   70413
Pasta           

#Insights:
in this prediction model, we can input the product parameters to predict the sales in the future. This model strongly support the idea that we should eliminate some bad products.